# Gap Prediction in Tech Stocks
### Can we predict when a stock will open significantly higher or lower than it closed?

A **gap** happens when the opening price the next morning is more than 1% away from yesterday's closing price.  
This usually happens after earnings announcements, major news, or macro events that come out overnight.  

**Our goal:** build a dataset, train ML models, and run experiments to understand what signals predict these gaps.

## Modules that we use

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble        
import RandomForestClassifier
from sklearn.linear_model    
import LogisticRegression
from sklearn.preprocessing   
import StandardScaler
from sklearn.model_selection 
import TimeSeriesSplit
from sklearn.metrics         
import (accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,confusion_matrix, roc_curve)
from sklearn.decomposition   import PCA
from imblearn.over_sampling  import SMOTE
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

## Analysis of 10 tech stocks over a 3 year period

In [2]:
TICKERS = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META',
           'TSLA', 'AMD',  'INTC', 'ASML',  'TSM']

START_DATE = '2021-01-01'   # 3 years of data
END_DATE   = '2024-01-01'
GAP_THRESH = 0.01           # gap = opening price moves more than 1% from yesterday's close

## Technical indicators implementation (simple and practical signals)

In [3]:
def compute_rsi(series, period=14):
    #RSI = tells you if a stock is overbought or oversold
    #If it's above ~70 --> probably overbought (price went up too fast)
    #If it's below ~30 --> probably oversold (price dropped too much)
    #Basically helps spot potential reversals

    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    rsi      = 100 - (100 / (1 + rs))
    return rsi


def compute_macd(series, fast=12, slow=26, signal=9):
    #MACD = tracks trend + momentum
    #It compares short-term vs long-term trend
    #When MACD crosses above signal --> bullish signal
    #When it crosses below --> bearish
    #Histogram shows how strong the move is

    ema_fast    = series.ewm(span=fast,   adjust=False).mean()
    ema_slow    = series.ewm(span=slow,   adjust=False).mean()
    macd_line   = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram   = macd_line - signal_line
    return macd_line, signal_line, histogram


def compute_bollinger_bands(series, period=20, std_dev=2):
    #Bollinger Bands shows volatility range
    #Price usually stays inside the bands
    #If it touches upper band --> maybe too high
    #If it touches lower band --> maybe too low
    #%B tells where price sits between the bands (0 = bottom, 1 = top)

    sma   = series.rolling(window=period).mean()
    std   = series.rolling(window=period).std()
    upper = sma + std_dev * std
    lower = sma - std_dev * std
    pct_b = (series - lower) / (upper - lower + 1e-9)  # normalized position in the bands
    return upper, lower, pct_b


def compute_atr(high, low, close, period=14):
    #ATR measures how much price moves (volatility)
    #High ATR = big moves --> more risk, and more likely to gap
    #Low ATR = calm market
    #Doesn't tell direction just how strong moves are

    tr  = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    atr = tr.rolling(window=period).mean()
    return atr


def compute_stochastic(high, low, close, period=14):
    #Stochastic tells where current price sits vs recent highs/lows
    #Near 100 --> price is close to recent highs
    #Near 0 --> close to recent lows
    #Good to detect overbought or oversold zones

    lowest_low   = low.rolling(window=period).min()
    highest_high = high.rolling(window=period).max()
    stoch_k      = 100 * (close - lowest_low) / (highest_high - lowest_low + 1e-9)
    stoch_d      = stoch_k.rolling(window=3).mean()   # smoothed version
    return stoch_k, stoch_d


def compute_volume_ratio(volume, period=20):
    #Volume ratio compares today's volume to the average
    #>1 --> more activity than usual
    #>1.5 --> big spike (something is happening)
    #Useful to confirm if a move is legit or weak

    avg_vol = volume.rolling(window=period).mean()
    return volume / (avg_vol + 1e-9)

## Building the dataset from raw prices to gap prediction features

In [4]:
all_dfs = []    # we store each stock's data here before merging everything

for ticker in TICKERS:
    print(f"{ticker}")

    # download raw market data (price and volume)
    raw = yf.download(ticker, start=START_DATE, end=END_DATE,
                      auto_adjust=True)

    # sometimes yfinance gives weird multi-level columns --> flatten them
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    # keep only the basic stuff we need
    df = pd.DataFrame({
        'open'   : raw['Open'],
        'high'   : raw['High'],
        'low'    : raw['Low'],
        'close'  : raw['Close'],
        'volume' : raw['Volume'],
    })
    df.index = pd.to_datetime(df.index)

    # Indicators
    # here we transform raw prices into signals traders actually use
    df['rsi_14'] = compute_rsi(df['close'], period=14)

    macd, macd_sig, macd_hist = compute_macd(df['close'])
    df['macd']        = macd        # trend direction
    df['macd_signal'] = macd_sig    # trigger line
    df['macd_hist']   = macd_hist   # strength of the move

    bb_up, bb_low, pct_b = compute_bollinger_bands(df['close'])
    df['bb_upper'] = bb_up          # upper volatility boundary
    df['bb_lower'] = bb_low         # lower boundary
    df['bb_pct_b'] = pct_b          # where price sits in the range

    df['atr_14'] = compute_atr(df['high'], df['low'], df['close'])  # how big moves are

    stoch_k, stoch_d = compute_stochastic(df['high'], df['low'], df['close'])
    df['stoch_k'] = stoch_k         # fast signal
    df['stoch_d'] = stoch_d         # smoothed signal

    # this part captures momentum + risk
    df['return_1d']      = df['close'].pct_change(1)          # daily return
    df['return_5d']      = df['close'].pct_change(5)          # short-term momentum
    df['volatility_5d']  = df['return_1d'].rolling(5).std()   # short-term risk
    df['volatility_20d'] = df['return_1d'].rolling(20).std()  # medium-term risk
    df['volume_ratio']   = compute_volume_ratio(df['volume']) # unusual activity detector

    # gap-specific features
    # these capture how the stock behaves AT the open, not during the day
    df['prev_gap_pct']  = (df['open'] - df['close'].shift(1)) / df['close'].shift(1)  # did it gap yesterday?
    df['close_to_high'] = (df['high'] - df['close']) / df['close']   # how far close is from day's high
    df['close_to_low']  = (df['close'] - df['low'])  / df['close']   # how far close is from day's low
    # if stock closes near its low --> finished weak --> might gap down next morning

    # this is what we try to predict
    # 1 --> significant gap (opening price moves more than 1% from yesterday's close)
    # 0 --> normal open, no gap
    next_open     = df['open'].shift(-1)
    gap_pct       = (next_open - df['close']) / df['close']
    df['label']   = (gap_pct.abs() > GAP_THRESH).astype(int)
    df['gap_pct'] = gap_pct   # keep the raw value for analysis

    df['ticker'] = ticker  # so we know which row belongs to which stock

    all_dfs.append(df)
    print(f"     {len(df)} days of data from {ticker}")

AAPL


[*********************100%***********************]  1 of 1 completed


     753 days of data from AAPL
MSFT


[*********************100%***********************]  1 of 1 completed


     753 days of data from MSFT
NVDA


...

     753 days of data from INTC
ASML


[*********************100%***********************]  1 of 1 completed


     753 days of data from ASML
TSM


[*********************100%***********************]  1 of 1 completed

     753 days of data from TSM


## Merging everything and cleaning the dataset

In [5]:
print("Fusion of all stocks")
dataset = pd.concat(all_dfs, axis=0)  # stack all stocks together into one big dataset
dataset.index.name = 'date'
dataset = dataset.reset_index()       # bring date back as a normal column

# drop NaNs created by indicators (they need some past data to work)
# for example MACD needs ~26 days --> so early rows are incomplete
# also drop last row per stock because we have no next_open for it
n_before = len(dataset)
dataset  = dataset.dropna()           # keep only clean / usable rows
n_after  = len(dataset)

print(f"Lines before cleaning : {n_before}")
print(f"Lines after  cleaning : {n_after} ( deleted {n_before - n_after} NaN)")

Fusion of all stocks
Lines before cleaning : 7530
Lines after  cleaning : 7320 ( deleted 210 NaN)


## Check and Dataset Export

In [6]:
print("Label distribution (gap vs normal)")
label_counts = dataset['label'].value_counts()
n_gap   = label_counts.get(1, 0)
n_nogap = label_counts.get(0, 0)
print(f"  Gap day (1)  : {n_gap} ({n_gap/len(dataset)*100:.1f}%)")
print(f"  Normal  (0)  : {n_nogap} ({n_nogap/len(dataset)*100:.1f}%)")
# note: imbalanced dataset (~20% gaps) --> SMOTE will be useful later

print("\nGap size stats (gap days only)")
gap_days = dataset[dataset['label'] == 1]
print(f"  Average gap : {gap_days['gap_pct'].abs().mean()*100:.2f}%")
print(f"  Max gap     : {gap_days['gap_pct'].abs().max()*100:.2f}%")

print("\nQuick look at the dataset")
print(dataset[['date', 'ticker', 'close', 'gap_pct', 'atr_14',
               'volume_ratio', 'prev_gap_pct', 'label']].head(10).to_string())

# save the dataset to a csv file so we can reuse it later
output_file = 'prices_with_indicators.csv'
dataset.to_csv(output_file, index=False)

print(f"\nDataset saved to '{output_file}'")
print(f"Shape: {dataset.shape[0]} rows x {dataset.shape[1]} columns")
print(f"Columns: {list(dataset.columns)}")

Label distribution (gap vs normal)
  Gap day (1)  : 2700 (36.9%)
  Normal  (0)  : 4620 (63.1%)

Gap size stats (gap days only)
  Average gap : 2.07%
  Max gap     : 26.15%

Quick look at the dataset
         date ticker       close   gap_pct    atr_14  volume_ratio  prev_gap_pct  label
20 2021-02-02   AAPL  131.283508  0.005704  4.275711      0.728670      0.011853      0
21 2021-02-03   AAPL  130.262299  0.017620  4.220138      0.788871      0.005704      1
22 2021-02-04   AAPL  133.617569  0.001203  4.304887      0.762591      0.017620      0
23 2021-02-05   AAPL  133.203629 -0.005338  4.189733      0.696377      0.001203      0
24 2021-02-08   AAPL  133.349701 -0.002118  4.208701      0.666307     -0.005338      0
25 2021-02-09   AAPL  132.473145  0.003456  4.026213      0.725496     -0.002118      0
26 2021-02-10   AAPL  131.869263  0.003767  3.675675      0.696492      0.003456      0
27 2021-02-11   AAPL  131.616013 -0.005772  3.522424      0.620104      0.003767      0
28 2021-0

## Feature list for the models
We only feed engineered features to the models, not raw prices.  
Raw prices (open, high, low, close) would just teach the model the price level, not the pattern.

In [7]:
FEATURE_COLUMNS = [
    'rsi_14',           # momentum oscillator
    'macd',             # trend direction
    'macd_signal',      # trigger line
    'macd_hist',        # strength of the move
    'bb_pct_b',         # where price sits in Bollinger bands
    'atr_14',           # absolute volatility --> gaps correlate with high ATR
    'stoch_k',          # stochastic fast signal
    'stoch_d',          # stochastic smoothed signal
    'return_1d',        # daily return
    'return_5d',        # short-term momentum
    'volatility_5d',    # short-term risk
    'volatility_20d',   # medium-term risk
    'volume_ratio',     # unusual volume --> often signals upcoming gaps
    'prev_gap_pct',     # did it gap yesterday? gaps tend to cluster
    'close_to_high',    # how far close is from the day's high
    'close_to_low',     # how far close is from the day's low
]

X = dataset[FEATURE_COLUMNS].values
y = dataset['label'].values

print(f"{len(FEATURE_COLUMNS)} features ready")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# save feature list so we can reload it later without rerunning everything
with open('feature_columns.txt', 'w') as f:
    f.write('\n'.join(FEATURE_COLUMNS))
print("Feature list saved to feature_columns.txt")

16 features ready
X shape: (7320, 16)
y shape: (7320,)
Feature list saved to feature_columns.txt


## Model training and evaluation
We test 3 models to compare a simple linear baseline vs ensemble methods.  
We use **TimeSeriesSplit** for cross-validation because random splits would let the model train on the future and test on the past --> that's cheating in finance.

In [8]:
N_SPLITS = 5
tscv     = TimeSeriesSplit(n_splits=N_SPLITS)

# class imbalance ratio for XGBoost
# XGBoost has a scale_pos_weight parameter to handle imbalanced classes
# it basically tells the model: pay more attention to the rare class
n_gap_total = y.sum()
n_total     = len(y)
pos_weight  = (n_total - n_gap_total) / n_gap_total

print(f"Cross-validation : TimeSeriesSplit {N_SPLITS} folds")
print(f"Gap days   : {n_gap_total} ({n_gap_total/n_total*100:.1f}%)")
print(f"Normal days: {n_total - n_gap_total} ({(n_total-n_gap_total)/n_total*100:.1f}%)")
print(f"scale_pos_weight for XGBoost: {pos_weight:.2f}")

Cross-validation : TimeSeriesSplit 5 folds
Gap days   : 2700 (36.9%)
Normal days: 4620 (63.1%)
scale_pos_weight for XGBoost: 1.71


In [9]:
MODELS = {
    'Logistic Regression (baseline)': LogisticRegression(
        max_iter=1000, C=1.0, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=15,
        class_weight='balanced',   # handles imbalance natively
        random_state=42,
        n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=pos_weight,   # tells XGBoost gaps are rare but important
        eval_metric='logloss',
        random_state=42,
        verbosity=0
    ),
}

all_results     = {}
all_predictions = {}

for model_name, model in MODELS.items():
    print(f"Training: {model_name}")

    fold_metrics = {'accuracy': [], 'f1': [], 'precision': [], 'recall': [], 'auroc': []}
    y_true_all, y_pred_all, y_proba_all = [], [], []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # normalize -- fit on train only so validation stays unseen
        scaler  = StandardScaler()
        X_tr    = scaler.fit_transform(X_tr)
        X_val   = scaler.transform(X_val)

        model.fit(X_tr, y_tr)
        y_pred  = model.predict(X_val)
        y_proba = model.predict_proba(X_val)[:, 1]

        fold_metrics['accuracy'].append(accuracy_score(y_val, y_pred))
        fold_metrics['f1'].append(f1_score(y_val, y_pred, zero_division=0))
        fold_metrics['precision'].append(precision_score(y_val, y_pred, zero_division=0))
        fold_metrics['recall'].append(recall_score(y_val, y_pred, zero_division=0))
        fold_metrics['auroc'].append(roc_auc_score(y_val, y_proba))

        y_true_all.extend(y_val)
        y_pred_all.extend(y_pred)
        y_proba_all.extend(y_proba)

        print(f"     Fold {fold+1}/{N_SPLITS} -- Acc={fold_metrics['accuracy'][-1]:.3f}  F1={fold_metrics['f1'][-1]:.3f}  AUROC={fold_metrics['auroc'][-1]:.3f}")

    all_results[model_name] = {
        'Accuracy'  : f"{np.mean(fold_metrics['accuracy']):.3f} +/- {np.std(fold_metrics['accuracy']):.3f}",
        'F1'        : f"{np.mean(fold_metrics['f1']):.3f} +/- {np.std(fold_metrics['f1']):.3f}",
        'Precision' : f"{np.mean(fold_metrics['precision']):.3f} +/- {np.std(fold_metrics['precision']):.3f}",
        'Recall'    : f"{np.mean(fold_metrics['recall']):.3f} +/- {np.std(fold_metrics['recall']):.3f}",
        'AUROC'     : f"{np.mean(fold_metrics['auroc']):.3f} +/- {np.std(fold_metrics['auroc']):.3f}",
    }
    all_predictions[model_name] = {
        'y_true' : np.array(y_true_all),
        'y_pred' : np.array(y_pred_all),
        'y_proba': np.array(y_proba_all),
    }
    print(f"     --> Mean AUROC: {np.mean(fold_metrics['auroc']):.3f}\n")

print("Results (mean +/- std over 5 folds)")
results_df = pd.DataFrame(all_results).T
print(results_df.to_string())
results_df.to_csv('results_summary.csv')

Training: Logistic Regression (baseline)
     Fold 1/5 -- Acc=0.643  F1=0.596  AUROC=0.711
     Fold 2/5 -- Acc=0.601  F1=0.499  AUROC=0.642
     Fold 3/5 -- Acc=0.596  F1=0.691  AUROC=0.681
     Fold 4/5 -- Acc=0.582  F1=0.426  AUROC=0.568
     Fold 5/5 -- Acc=0.600  F1=0.548  AUROC=0.628
     --> Mean AUROC: 0.646

Training: Random Forest
     Fold 1/5 -- Acc=0.661  F1=0.487  AUROC=0.698
     Fold 2/5 -- Acc=0.619  F1=0.508  AUROC=0.657
     Fold 3/5 -- Acc=0.594  F1=0.688  AUROC=0.662
     Fold 4/5 -- Acc=0.571  F1=0.468  AUROC=0.586
     Fold 5/5 -- Acc=0.592  F1=0.559  AUROC=0.632
     --> Mean AUROC: 0.647

Training: XGBoost
     Fold 1/5 -- Acc=0.653  F1=0.287  AUROC=0.673
     Fold 2/5 -- Acc=0.627  F1=0.420  AUROC=0.643
     Fold 3/5 -- Acc=0.575  F1=0.619  AUROC=0.614
     Fold 4/5 -- Acc=0.568  F1=0.476  AUROC=0.599
     Fold 5/5 -- Acc=0.601  F1=0.561  AUROC=0.625
     --> Mean AUROC: 0.631

Results (mean +/- std over 5 folds)
                                       Accuracy

## Figures — ROC curves, confusion matrices, feature importance

In [10]:
# ROC curves -- one per model
# AUROC = area under this curve, higher is better, 0.5 = random
fig, ax = plt.subplots(figsize=(8, 6))
colors  = ['#3b82f6', '#f59e0b', '#10b981']

for (model_name, preds), color in zip(all_predictions.items(), colors):
    fpr, tpr, _ = roc_curve(preds['y_true'], preds['y_proba'])
    auc = roc_auc_score(preds['y_true'], preds['y_proba'])
    ax.plot(fpr, tpr, label=f"{model_name} (AUROC={auc:.3f})", color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random baseline')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves -- Gap Day Prediction (5-fold TimeSeriesSplit)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: roc_curve.png")

Saved: roc_curve.png


In [11]:
# confusion matrices -- shows what the model gets right and wrong
# rows = actual class, columns = predicted class
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (model_name, preds) in zip(axes, all_predictions.items()):
    cm     = confusion_matrix(preds['y_true'], preds['y_pred'])
    cm_pct = cm / cm.sum()
    ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=0.7)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Predicted: Normal', 'Predicted: Gap'], fontsize=8)
    ax.set_yticklabels(['Actual: Normal', 'Actual: Gap'], fontsize=8)
    ax.set_title(model_name, fontsize=10, fontweight='bold')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]}\n({cm_pct[i,j]:.1%})",
                    ha='center', va='center', fontsize=11,
                    color='white' if cm_pct[i,j] > 0.4 else 'black')

plt.suptitle('Confusion Matrices -- 5 folds combined', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: confusion_matrices.png")

Saved: confusion_matrices.png


In [12]:
# feature importance from Random Forest
# tells us which features the model relies on the most
# higher = more useful for predicting gaps
scaler_full = StandardScaler()
X_sc        = scaler_full.fit_transform(X)
rf_full     = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=15,
                                     class_weight='balanced', random_state=42, n_jobs=-1)
rf_full.fit(X_sc, y)
importances = rf_full.feature_importances_
sorted_idx  = np.argsort(importances)   # sort from lowest to highest

# color gap-specific features differently to highlight them
gap_features = {'prev_gap_pct', 'close_to_high', 'close_to_low', 'atr_14'}
colors_bar   = ['#ef4444' if FEATURE_COLUMNS[i] in gap_features else '#3b82f6'
                for i in sorted_idx]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(len(FEATURE_COLUMNS)), importances[sorted_idx], color=colors_bar)
ax.set_yticks(range(len(FEATURE_COLUMNS)))
ax.set_yticklabels([FEATURE_COLUMNS[i] for i in sorted_idx], fontsize=10)
ax.set_xlabel('Importance (Gini)', fontsize=12)
ax.set_title('Feature Importance -- Random Forest\n(red = gap-specific features)', fontsize=12)
ax.grid(axis='x', alpha=0.3)
for bar, idx in zip(ax.patches, sorted_idx):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{importances[idx]:.3f}", va='center', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: feature_importance.png")

Saved: feature_importance.png


## Experiment 1 — Does more training data improve gap prediction?
We train on increasing fractions of the dataset (10% to 100%) and measure if performance improves.  
If it plateaus early --> the signal is weak and data-hungry models won't help.

In [13]:
def quick_eval(model, X, y, tscv, use_smote=False):
    # runs the model through all CV folds and returns average metrics
    # use_smote=True --> creates synthetic gap examples in training fold only
    accs, f1s, aurocs, precs, recs = [], [], [], [], []
    for train_idx, val_idx in tscv.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        scaler = StandardScaler()
        X_tr   = scaler.fit_transform(X_tr)
        X_val  = scaler.transform(X_val)
        if use_smote and len(np.unique(y_tr)) > 1:
            try:
                sm     = SMOTE(random_state=42, k_neighbors=min(5, int(y_tr.sum()) - 1))
                X_tr, y_tr = sm.fit_resample(X_tr, y_tr)
            except Exception:
                pass   # if SMOTE fails on small datasets just skip it
        model.fit(X_tr, y_tr)
        y_pred  = model.predict(X_val)
        y_proba = model.predict_proba(X_val)[:, 1]
        accs.append(accuracy_score(y_val, y_pred))
        f1s.append(f1_score(y_val, y_pred, zero_division=0))
        aurocs.append(roc_auc_score(y_val, y_proba))
        precs.append(precision_score(y_val, y_pred, zero_division=0))
        recs.append(recall_score(y_val, y_pred, zero_division=0))
    return {'acc': np.mean(accs), 'f1': np.mean(f1s), 'auroc': np.mean(aurocs),
            'prec': np.mean(precs), 'rec': np.mean(recs)}


# model factories -- fresh model at each call avoids state contamination between experiments
def RF():
    return RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=15,
                                  class_weight='balanced', random_state=42, n_jobs=-1)
def XGB():
    return xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05,
                             scale_pos_weight=pos_weight,
                             eval_metric='logloss', random_state=42, verbosity=0)


fractions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
exp1_rf, exp1_xgb = [], []

for frac in fractions:
    n            = max(200, int(len(X) * frac))
    X_sub, y_sub = X[:n], y[:n]
    tscv_sub     = TimeSeriesSplit(n_splits=min(N_SPLITS, max(2, n // 100)))
    r_rf         = quick_eval(RF(),  X_sub, y_sub, tscv_sub)
    r_xgb        = quick_eval(XGB(), X_sub, y_sub, tscv_sub)
    exp1_rf.append(r_rf)
    exp1_xgb.append(r_xgb)
    print(f"  {frac*100:4.0f}% ({n:5d} rows) -- RF={r_rf['auroc']:.3f}  XGB={r_xgb['auroc']:.3f}")

n_samples = [max(200, int(len(X) * f)) for f in fractions]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(n_samples, [r['auroc'] for r in exp1_rf],  'o-',  color='#3b82f6', lw=2, ms=7, label='Random Forest')
ax1.plot(n_samples, [r['auroc'] for r in exp1_xgb], 's--', color='#f59e0b', lw=2, ms=7, label='XGBoost')
ax1.axhline(0.5, color='gray', ls=':', lw=1, label='Random baseline')
ax1.set_xlabel('Number of training rows', fontsize=11)
ax1.set_ylabel('AUROC (mean, 5 folds)', fontsize=11)
ax1.set_title('AUROC vs training set size', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(n_samples, [r['f1'] for r in exp1_rf],  'o-',  color='#3b82f6', lw=2, ms=7, label='Random Forest')
ax2.plot(n_samples, [r['f1'] for r in exp1_xgb], 's--', color='#f59e0b', lw=2, ms=7, label='XGBoost')
ax2.set_xlabel('Number of training rows', fontsize=11)
ax2.set_ylabel('F1-score (mean, 5 folds)', fontsize=11)
ax2.set_title('F1-score vs training set size', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Experiment 1 -- Impact of training data size on gap prediction\nQuestion: Does more data improve performance or does it plateau early?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('exp1_data_size.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: exp1_data_size.png")

    10% (  732 rows) -- RF=0.614  XGB=0.554
    20% ( 1464 rows) -- RF=0.625  XGB=0.604
    30% ( 2196 rows) -- RF=0.657  XGB=0.636
    40% ( 2928 rows) -- RF=0.668  XGB=0.653
    50% ( 3660 rows) -- RF=0.685  XGB=0.665
    60% ( 4392 rows) -- RF=0.667  XGB=0.642
    70% ( 5124 rows) -- RF=0.673  XGB=0.654
    80% ( 5856 rows) -- RF=0.643  XGB=0.630
    90% ( 6588 rows) -- RF=0.651  XGB=0.636
   100% ( 7320 rows) -- RF=0.645  XGB=0.630
Saved: exp1_data_size.png


## Experiment 2 — SMOTE vs class weighting on imbalanced data
Gap days are only ~20% of our dataset.  
We test 3 strategies: no resampling, class_weight=balanced, and SMOTE.  
Unlike direction prediction (50/50 split), the imbalance here is real so resampling should actually help.

In [14]:
configs = {
    'No resampling'         : dict(use_smote=False, cw=None),
    'class_weight=balanced' : dict(use_smote=False, cw='balanced'),
    'SMOTE'                 : dict(use_smote=True,  cw=None),
}

exp2_results = {}
for config_name, params in configs.items():
    rf = RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=15,
                                class_weight=params['cw'], random_state=42, n_jobs=-1)
    r  = quick_eval(rf, X, y, tscv, use_smote=params['use_smote'])
    exp2_results[config_name] = r
    print(f"  {config_name:30s} -- F1={r['f1']:.3f}  Recall={r['rec']:.3f}  AUROC={r['auroc']:.3f}")

metrics_show  = ['acc', 'f1', 'prec', 'rec', 'auroc']
metric_labels = ['Accuracy', 'F1', 'Precision', 'Recall', 'AUROC']
x  = np.arange(len(metrics_show))
bw = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
colors_exp2 = ['#6b7280', '#3b82f6', '#10b981']
for i, (config_name, r) in enumerate(exp2_results.items()):
    vals = [r[m] for m in metrics_show]
    ax.bar(x + i*bw, vals, bw, label=config_name, color=colors_exp2[i], alpha=0.85)
ax.set_xticks(x + bw)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_title('Experiment 2 -- SMOTE vs class_weight vs nothing\nQuestion: Does resampling help when gaps are ~20% of data?',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('exp2_imbalance.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: exp2_imbalance.png")

  No resampling                  -- F1=0.277  Recall=0.205  AUROC=0.648
  class_weight=balanced          -- F1=0.546  Recall=0.628  AUROC=0.645
  SMOTE                          -- F1=0.535  Recall=0.619  AUROC=0.639
Saved: exp2_imbalance.png


## Experiment 3 — Gap-specific features vs standard features only
We added 3 features specifically for gap prediction: `prev_gap_pct`, `close_to_high`, `close_to_low`.  
This experiment checks if they actually help or if standard indicators are enough.

In [15]:
# standard features = everything except the 3 gap-specific ones
standard_features = [f for f in FEATURE_COLUMNS if f not in ('prev_gap_pct', 'close_to_high', 'close_to_low')]
X_standard = dataset[standard_features].values
X_gap_all  = X   # full feature set including gap-specific ones

exp3_results = {}
for name, X_exp in [('Standard features only', X_standard),
                     ('+ Gap-specific features', X_gap_all)]:
    r_rf  = quick_eval(RF(),  X_exp, y, tscv)
    r_xgb = quick_eval(XGB(), X_exp, y, tscv)
    exp3_results[name] = {'RF': r_rf, 'XGB': r_xgb}
    print(f"  {name}")
    print(f"     RF  -- AUROC={r_rf['auroc']:.3f}  F1={r_rf['f1']:.3f}")
    print(f"     XGB -- AUROC={r_xgb['auroc']:.3f}  F1={r_xgb['f1']:.3f}")

categories = list(exp3_results.keys())
rf_aurocs  = [exp3_results[c]['RF']['auroc']  for c in categories]
xgb_aurocs = [exp3_results[c]['XGB']['auroc'] for c in categories]
rf_f1s     = [exp3_results[c]['RF']['f1']     for c in categories]
xgb_f1s    = [exp3_results[c]['XGB']['f1']    for c in categories]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
x, bw = np.arange(len(categories)), 0.35

ax1.bar(x - bw/2, rf_aurocs,  bw, label='Random Forest', color='#3b82f6', alpha=0.85)
ax1.bar(x + bw/2, xgb_aurocs, bw, label='XGBoost',       color='#f59e0b', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(categories, fontsize=9)
ax1.set_ylabel('AUROC', fontsize=11)
ax1.set_ylim(0.45, 0.80)
ax1.axhline(0.5, color='gray', ls=':', lw=1)
ax1.set_title('AUROC: standard vs + gap features', fontsize=11)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

ax2.bar(x - bw/2, rf_f1s,  bw, label='Random Forest', color='#3b82f6', alpha=0.85)
ax2.bar(x + bw/2, xgb_f1s, bw, label='XGBoost',       color='#f59e0b', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(categories, fontsize=9)
ax2.set_ylabel('F1-score', fontsize=11)
ax2.set_ylim(0.0, 0.8)
ax2.set_title('F1: standard vs + gap features', fontsize=11)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Experiment 3 -- Do gap-specific features improve prediction?\nQuestion: Are prev_gap_pct, close_to_high, close_to_low actually useful?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('exp3_gap_features.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: exp3_gap_features.png")

  Standard features only
     RF  -- AUROC=0.640  F1=0.540
     XGB -- AUROC=0.630  F1=0.493
  + Gap-specific features
     RF  -- AUROC=0.645  F1=0.546
     XGB -- AUROC=0.630  F1=0.494
Saved: exp3_gap_features.png


## Experiment 4 — PCA dimensionality reduction
We have 16 features. PCA compresses them into fewer dimensions.  
If performance stays the same with fewer components --> features are redundant.  
If it drops --> every feature carries unique information.

In [16]:
n_components_list = [3, 5, 7, 10, 12, len(FEATURE_COLUMNS)]
labels_list       = [str(n) if n < len(FEATURE_COLUMNS) else 'All\n(no PCA)' for n in n_components_list]
exp4_rf, exp4_xgb = [], []

for n_comp in n_components_list:
    if n_comp == len(FEATURE_COLUMNS):
        X_pca = X   # no PCA -- use raw features
    else:
        scaler_pca = StandardScaler()
        X_sc       = scaler_pca.fit_transform(X)
        pca        = PCA(n_components=n_comp, random_state=42)
        X_pca      = pca.fit_transform(X_sc)
    r_rf  = quick_eval(RF(),  X_pca, y, tscv)
    r_xgb = quick_eval(XGB(), X_pca, y, tscv)
    exp4_rf.append(r_rf['auroc'])
    exp4_xgb.append(r_xgb['auroc'])
    print(f"  {n_comp} components -- RF={r_rf['auroc']:.3f}  XGB={r_xgb['auroc']:.3f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(n_components_list)), exp4_rf,  'o-',  color='#3b82f6', lw=2, ms=8, label='Random Forest')
ax.plot(range(len(n_components_list)), exp4_xgb, 's--', color='#f59e0b', lw=2, ms=8, label='XGBoost')
ax.set_xticks(range(len(n_components_list)))
ax.set_xticklabels(labels_list, fontsize=10)
ax.set_ylabel('AUROC (mean, 5 folds)', fontsize=11)
ax.set_xlabel('PCA components', fontsize=11)
ax.set_title('Experiment 4 -- PCA dimensionality reduction\nQuestion: Are our 16 features redundant or does each one matter?',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('exp4_pca.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: exp4_pca.png")

  3 components -- RF=0.647  XGB=0.640
  5 components -- RF=0.655  XGB=0.643
  7 components -- RF=0.646  XGB=0.640
  10 components -- RF=0.645  XGB=0.634
  12 components -- RF=0.646  XGB=0.638
  16 components -- RF=0.645  XGB=0.630
Saved: exp4_pca.png


## Experiment 5 — Confidence threshold: predict less but predict better
Instead of predicting every day, we only predict when the model is very confident.  
Higher threshold --> fewer predictions but more accurate.  
This is an original research question: can we trade precision for coverage?

In [17]:
# collect out-of-fold probabilities from Random Forest
all_y_true, all_y_proba = [], []
for train_idx, val_idx in tscv.split(X):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr)
    X_val  = scaler.transform(X_val)
    rf = RF()
    rf.fit(X_tr, y_tr)
    all_y_true.extend(y_val)
    all_y_proba.extend(rf.predict_proba(X_val)[:, 1])

all_y_true  = np.array(all_y_true)
all_y_proba = np.array(all_y_proba)

# sweep from 0.50 (predict everything) to 0.75 (predict only very confident cases)
thresholds           = np.arange(0.50, 0.76, 0.01)
precisions, coverages = [], []

for thresh in thresholds:
    mask = all_y_proba >= thresh   # only keep high-confidence gap predictions
    if mask.sum() < 20:
        break   # too few predictions to be reliable
    prec = precision_score(all_y_true[mask], (all_y_proba[mask] >= 0.5).astype(int), zero_division=0)
    cov  = mask.sum() / len(all_y_true)
    precisions.append(prec)
    coverages.append(cov)

thresholds = thresholds[:len(precisions)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# left: main trade-off -- the higher we push precision, the fewer days we predict
ax1.plot(coverages, precisions, 'o-', color='#ef4444', lw=2, ms=6)
ax1.axhline(all_y_true.mean(), color='gray', ls='--',
            label=f'Actual gap rate ({all_y_true.mean():.1%})')
ax1.set_xlabel('Coverage (fraction of days we predict on)', fontsize=11)
ax1.set_ylabel('Precision (% of gap predictions correct)', fontsize=11)
ax1.set_title('Trade-off: Precision vs Coverage', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.invert_xaxis()   # left = selective, right = predict everything

# right: how precision evolves as we raise the bar
ax2.plot(thresholds * 100, precisions, 's-', color='#8b5cf6', lw=2, ms=6)
ax2.axhline(all_y_true.mean(), color='gray', ls=':', lw=1)
ax2.set_xlabel('Confidence threshold (%)', fontsize=11)
ax2.set_ylabel('Precision', fontsize=11)
ax2.set_title('Precision as threshold increases', fontsize=11)
ax2.grid(True, alpha=0.3)

plt.suptitle('Experiment 5 -- Confidence threshold: predict less, predict better\nQuestion: Can we improve precision by only predicting high-confidence gap days?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('exp5_confidence_threshold.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: exp5_confidence_threshold.png")

Saved: exp5_confidence_threshold.png


## Summary of all generated files

In [18]:
print("All files generated:")
files = [
    'prices_with_indicators.csv    --> the full dataset (7330 rows)',
    'feature_columns.txt           --> list of features used by the models',
    'results_summary.csv           --> model comparison table',
    'roc_curve.png                 --> ROC curves (main figure)',
    'confusion_matrices.png        --> confusion matrices',
    'feature_importance.png        --> which features matter most',
    'exp1_data_size.png            --> Exp 1: data size impact',
    'exp2_imbalance.png            --> Exp 2: SMOTE vs class_weight',
    'exp3_gap_features.png         --> Exp 3: gap-specific features',
    'exp4_pca.png                  --> Exp 4: PCA',
    'exp5_confidence_threshold.png --> Exp 5: confidence threshold',
]
for f in files:
    print(f"  {f}")

All files generated:
  prices_with_indicators.csv    --> the full dataset (7330 rows)
  feature_columns.txt           --> list of features used by the models
  results_summary.csv           --> model comparison table
  roc_curve.png                 --> ROC curves (main figure)
  confusion_matrices.png        --> confusion matrices
  feature_importance.png        --> which features matter most
  exp1_data_size.png            --> Exp 1: data size impact
  exp2_imbalance.png            --> Exp 2: SMOTE vs class_weight
  exp3_gap_features.png         --> Exp 3: gap-specific features
  exp4_pca.png                  --> Exp 4: PCA
  exp5_confidence_threshold.png --> Exp 5: confidence threshold
